In [0]:
%pip install gradio fastapi uvicorn sarvamai openai python-dotenv sentence-transformers faiss-cpu
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.9/542.9 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
# Load intelligence layer
def _load_notebook(path):
    code = open(path).read()
    lines = []
    for line in code.splitlines():
        stripped = line.strip()
        if stripped.startswith("# COMMAND ----------"):
            continue
        if stripped.startswith("%pip"):
            continue
        if stripped.startswith("%sh"):
            continue
        if "dbutils.library.restartPython" in stripped:
            continue
        lines.append(line)
    exec("\n".join(lines), globals())

_load_notebook("/Workspace/Shared/Sarkari-Mitra/intelligence/intelligence.py")
print("✅ Intelligence loaded")

✅ Environment loaded
✅ LLM client loaded
✅ Sarvam translation client loaded
✅ Constants loaded
✅ Router loaded


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index loaded — 18739 vectors
✅ Matcher loaded
✅ Action plan generator loaded
✅ Follow-up RAG loaded
✅ Intelligence layer fully loaded
✅ Intelligence loaded


In [0]:
text = "నేను ఆంధ్రప్రదేశ్‌కు చెందిన 23 సంవత్సరాల వయస్సు గల మహిళను"

# Check script scores
scores = {}
for lang, (start, end) in SCRIPT_RANGES.items():
    scores[lang] = sum(1 for c in text if start <= c <= end)
print(f"Script scores: {scores}")

# Check detected language
lang = detect_language(text)
print(f"Detected: {lang}")

# Check translation
translated, src = translate_to_english(text)
print(f"Source lang: {src}")
print(f"Translated: {translated}")

Script scores: {'hi': 0, 'te': 47, 'ta': 0, 'kn': 0, 'ml': 0, 'gu': 0, 'bn': 0, 'pa': 0, 'or': 0}
🔍 Script scores: {'hi': 0, 'te': 47, 'ta': 0, 'kn': 0, 'ml': 0, 'gu': 0, 'bn': 0, 'pa': 0, 'or': 0}
Detected: te
🔍 Script scores: {'hi': 0, 'te': 47, 'ta': 0, 'kn': 0, 'ml': 0, 'gu': 0, 'bn': 0, 'pa': 0, 'or': 0}
🌐 Detected language: te
✅ Sarvam translated te→en: I am a 23-year-old woman from Andhra Pradesh
Source lang: te
Translated: I am a 23-year-old woman from Andhra Pradesh


In [0]:
# Simulate exactly what respond() does for Telugu input
message = "నేను ఆంధ్రప్రదేశ్‌కు చెందిన 23 సంవత్సరాల వయస్సు గల మహిళను, నేను ఏ విద్యా పథకాలకి అర్హురాలిని"

session = {
    "profile":  EMPTY_PROFILE.copy(),
    "schemes":  [],
    "history":  [],
    "language": "en"
}

# Step 1
english_input, source_lang = translate_to_english(message)
print(f"source_lang: {source_lang}")
print(f"english_input: {english_input}")

if source_lang != "en":
    session["language"] = source_lang

print(f"session language after step 1: {session['language']}")

# Step 4 simulation
target_lang = session.get("language", "en")
print(f"target_lang: {target_lang}")

test_reply = "Hello this is a test reply"
if target_lang not in ("en", "hi"):
    translated = translate_response(test_reply, target_lang)
    print(f"Translated reply: {translated}")
else:
    print(f"NOT translating — target_lang is {target_lang}")

🔍 Script scores: {'hi': 0, 'te': 76, 'ta': 0, 'kn': 0, 'ml': 0, 'gu': 0, 'bn': 0, 'pa': 0, 'or': 0}
🌐 Detected language: te
✅ Sarvam translated te→en: I am a 23-year-old woman from Andhra Pradesh. Which educatio
source_lang: te
english_input: I am a 23-year-old woman from Andhra Pradesh. Which educational schemes am I eligible for
session language after step 1: te
target_lang: te
✅ Sarvam translated en→te
Translated reply: నమస్కారం, ఇది ఒక పరీక్షా ప్రత్యుత్తరం.


In [0]:
import gradio as gr
import __main__
import subprocess, time

subprocess.run(["pkill", "-f", "gradio"], capture_output=True)
time.sleep(2)

_required = [
    "translate_to_english", "translate_response",
    "route_and_extract", "hybrid_match",
    "generate_action_plan", "answer_followup",
    "get_clarifying_question", "EMPTY_PROFILE",
    "INTENT_GREETING", "INTENT_OFFTOPIC",
    "INTENT_UNCLEAR", "INTENT_FOLLOWUP", "INTENT_DETAIL",
    "INTENT_DISCOVERY", "SCRIPT_RANGES", "detect_language",
    "detect_language_switch", "LANGUAGE_NAMES"
]

all_ok = True
for _name in _required:
    try:
        globals()[_name] = getattr(__main__, _name)
        print(f"✅ {_name}")
    except AttributeError:
        print(f"❌ {_name} — missing, re-run Cell 2")
        all_ok = False

if not all_ok:
    raise RuntimeError("Some functions missing. Re-run Cell 2 first.")

session_store = {}

def get_or_create_session(sid):
    if sid not in session_store:
        session_store[sid] = {
            "profile":  EMPTY_PROFILE.copy(),
            "schemes":  [],
            "history":  [],
            "language": "en"
        }
    return session_store[sid]

def build_enriched_query(english_input: str, profile: dict) -> str:
    parts = [english_input]
    if profile.get("occupation"):         parts.append(f"{profile['occupation']} scheme")
    if profile.get("gender") == "female": parts.append("women scheme")
    if profile.get("age"):                parts.append(f"age {profile['age']}")
    if profile.get("state"):              parts.append(f"{profile['state']} government scheme")
    if profile.get("category"):           parts.append(f"{profile['category']} category scheme")
    if profile.get("has_bpl_card"):       parts.append("BPL below poverty line")
    if profile.get("specific_need"):      parts.append(f"{profile['specific_need']} scheme benefit")
    return " ".join(parts)

def do_discovery(english_input, session, source_lang):
    if not session["profile"].get("state"):
        return get_clarifying_question(session["profile"], source_lang) or "Which state are you from?"

    enriched_query = build_enriched_query(english_input, session["profile"])
    print(f"🔍 Enriched query: {enriched_query}")

    session["schemes"] = hybrid_match(
        profile    = session["profile"],
        user_input = enriched_query,
        top_n      = 10
    )
    print(f"📋 Found {len(session['schemes'])} schemes")

    if not session["schemes"]:
        if source_lang == "hi":
            return "माफ़ कीजिए, कोई योजना नहीं मिली। कृपया राज्य, काम, आय और जाति वर्ग बताएं।"
        return "Sorry, no schemes found. Please share your state, occupation, income and caste category."

    print("📝 Generating action plan...")
    return generate_action_plan(
        profile     = session["profile"],
        schemes     = session["schemes"],
        user_input  = english_input,
        source_lang = source_lang
    )

def respond(message, history):
    if not message.strip():
        return ""

    session_id = "default_user_session"
    session    = get_or_create_session(session_id)

    # ── Language switch check ──────────────────────────────
    detected_switch_lang = detect_language_switch(message)
    if detected_switch_lang:
        session["language"] = detected_switch_lang
        lang_name = LANGUAGE_NAMES.get(detected_switch_lang, detected_switch_lang)
        if session["history"]:
            last_reply = session["history"][-1]["content"]
            if detected_switch_lang not in ("en", "hi"):
                try:
                    last_reply = translate_response(last_reply, detected_switch_lang)
                except Exception as e:
                    print(f"⚠️ Switch translation error: {e}")
            return last_reply
        if detected_switch_lang == "hi":
            return "ठीक है! अब मैं हिंदी में जवाब दूंगा। आप क्या जानना चाहते हैं?"
        if detected_switch_lang == "en":
            return "Sure! I will now respond in English. How can I help you?"
        return f"Sure! Switching to {lang_name}. How can I help?"

    # ── Step 1: Detect language + translate to English ─────
    try:
        english_input, source_lang = translate_to_english(message)
        # Store detected language — this is the ONLY place we set it
        session["language"] = source_lang
        print(f"🌐 Lang: {source_lang} | Input: {english_input[:80]}")
    except Exception as e:
        print(f"⚠️ Translation error: {e}")
        english_input = message
        source_lang   = "en"
        session["language"] = "en"

    # ── Step 2: Route intent + extract profile ─────────────
    try:
        route_result       = route_and_extract(
            user_input           = english_input,
            existing_profile     = session["profile"],
            conversation_history = session["history"][-4:]
        )
        intent             = route_result.get("intent", INTENT_UNCLEAR)
        session["profile"] = route_result.get("profile", session["profile"])
        print(f"🎯 Intent: {intent}")
        print(f"👤 Profile: {session['profile']}")
    except Exception as e:
        return f"Sorry, something went wrong with routing: {e}"

    # ── Step 3: Generate reply ─────────────────────────────
    reply = ""
    try:
        if intent == INTENT_GREETING:
            if source_lang == "hi":
                reply = "नमस्ते! मैं सरकारी मित्र हूं 🙏\n\nमैं आपको सरकारी योजनाओं की जानकारी दे सकता हूं।\n\nबताइए:\n• आप किस राज्य से हैं?\n• आपका काम क्या है?\n• क्या आपके पास BPL कार्ड है?"
            else:
                reply = "Hello! I am Sarkari Mitra 🙏\n\nI help Indian citizens find government schemes.\n\nPlease tell me:\n• Which **state** are you from?\n• What is your **occupation**?\n• Do you have a **BPL card**?"

        elif intent == INTENT_OFFTOPIC:
            if source_lang == "hi":
                reply = "माफ़ कीजिए, मैं केवल सरकारी योजनाओं की जानकारी दे सकता हूं।"
            else:
                reply = "I can only help with Indian government schemes. Please tell me about yourself."

        elif intent == INTENT_UNCLEAR:
            reply = get_clarifying_question(session["profile"], source_lang) or (
                "आप किस राज्य से हैं?" if source_lang == "hi" else "Which state are you from?"
            )

        elif intent in (INTENT_FOLLOWUP, INTENT_DETAIL) and session["schemes"]:
            top   = session["schemes"][0]
            reply = answer_followup(
                question    = english_input,
                scheme_id   = top["scheme_id"],
                scheme_name = top["scheme_name"],
                source_lang = source_lang
            )

        else:
            # SCHEME_DISCOVERY or DETAIL/FOLLOWUP with no schemes yet
            reply = do_discovery(english_input, session, source_lang)

    except Exception as e:
        print(f"❌ Reply error: {e}")
        reply = f"Sorry, something went wrong: {e}"

    # ── Step 4: Translate back for non-English non-Hindi ───
    # English → already in English, no translation needed
    # Hindi → Llama already responded in Hindi, no translation needed
    # Telugu/Tamil/etc → translate English reply via Sarvam
    if source_lang not in ("en", "hi"):
        try:
            reply = translate_response(reply, source_lang)
            print(f"✅ Reply translated to: {source_lang}")
        except Exception as e:
            print(f"❌ Response translation FAILED: {e}")

    # ── Step 5: Save history ───────────────────────────────
    session["history"].append({"role": "user",      "content": message})
    session["history"].append({"role": "assistant",  "content": reply})
    session["history"] = session["history"][-20:]

    return reply


print("\n🚀 Launching Sarkari Mitra...")

try:
    gr.ChatInterface(
        fn=respond,
        title="🇮🇳 Sarkari Mitra",
        description="Government scheme advisor — Hindi, English, Tamil, Telugu, Marathi and more.",
        examples=[
            "मैं MP से हूं, किसान हूं, BPL कार्ड है",
            "I am from Bihar, SC student, age 19",
            "Women entrepreneur loan schemes in Maharashtra",
            "நான் தமிழ்நாட்டில் விவசாயி",
            "నేను తెలంగాణ నుండి రైతును"
        ]
    ).launch(
        share=True,
        show_error=True
    )
except Exception as e:
    print(f"❌ Launch failed: {e}")
    print("Try restarting your Databricks cluster from the Compute tab.")

✅ translate_to_english
✅ translate_response
✅ route_and_extract
✅ hybrid_match
✅ generate_action_plan
✅ answer_followup
✅ get_clarifying_question
✅ EMPTY_PROFILE
✅ INTENT_GREETING
✅ INTENT_OFFTOPIC
✅ INTENT_UNCLEAR
✅ INTENT_FOLLOWUP
✅ INTENT_DETAIL
✅ INTENT_DISCOVERY
✅ SCRIPT_RANGES
✅ detect_language
✅ detect_language_switch
✅ LANGUAGE_NAMES

🚀 Launching Sarkari Mitra...
* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://1ea9feaa17caea4f2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [0]:
result = respond("నేను ఆంధ్రప్రదేశ్‌కు చెందిన 23 సంవత్సరాల వయస్సు గల మహిళను, నేను ఏ విద్యా పథకాలకి అర్హురాలిని?", [])
print(f"REPLY LANGUAGE CHECK: {result[:100]}")

🔍 Script scores: {'hi': 0, 'te': 76, 'ta': 0, 'kn': 0, 'ml': 0, 'gu': 0, 'bn': 0, 'pa': 0, 'or': 0}
🌐 Detected language: te
✅ Sarvam translated te→en: I am a 23-year-old woman from Andhra Pradesh. Which educatio
🌐 Lang: te | Input: I am a 23-year-old woman from Andhra Pradesh. Which educational schemes am I eli
🎯 Intent: SCHEME_DETAIL
👤 Profile: {'state': 'Andhra Pradesh', 'age': 23, 'gender': 'female', 'income_annual': None, 'category': None, 'occupation': None, 'has_bpl_card': None, 'specific_need': 'education', 'clarify_field': 'income_annual'}
🔄 target_lang: te
✅ Sarvam translated en→te
✅ Reply translated to: te
REPLY LANGUAGE CHECK: అందించిన సందర్భంలో విద్యా పథకాల గురించి ఎటువంటి సమాచారం లేదు. ఈ సందర్భం ప్రత్యేకంగా వికలాంగులు (పిడబ


In [0]:
import subprocess
result = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "https://api.gradio.app"],
    capture_output=True, text=True
)
print(f"Gradio API status: {result.stdout}")

result2 = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "https://huggingface.co"],
    capture_output=True, text=True
)
print(f"HuggingFace status: {result2.stdout}")

Gradio API status: 200
HuggingFace status: 200
